# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilalFarooq/Assignment1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Can early-month search signals (impressions, clicks, position) and static content
attributes (age, length) identify which content pages are likely to be declining in visibility,
well enough to beat a simple, transparent rule?

**Decision this supports:** a content team managing a large portfolio cannot manually review every
page every month. This work supports a triage decision -- which pages a human reviewer should look
at first when deciding where to spend limited refresh/review effort -- not an automated content
action system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank internship warehouse (Hugging Face, hf://datasets/FlyRank/internship-warehouse).

**Tables used:**
- `fact_content_daily_performance`, March 2026 partition only (month=2026-03) -- 9.84M rows, 55
  clients, 331K content items, all report_dates in March 2026.
- `dim_content` -- static content attributes (content type, word/char count, backlinks, search
  volume, competition, main intent, category count, content creation date).

**Date windows:** early-March (days 1-15) used to build features; late-March (days 16-31) used to
define the outcome label. All feature data strictly precedes the outcome window.

**Excluded, and why:**
- `fact_content_daily_performance_sample` -- the sealed final month (June 2026), reserved as a
  held-out test month and never touched for label logic.
- `fact_content_query_90d` -- its actual window covers April-June 2026 only. It does not reach back
  to March, so it cannot honestly inform a March-based label.
- `trend_direction` / `trend_pct` -- excluded entirely to avoid circularity with the label.
- All client names, URLs, and raw queries are excluded throughout -- only hashed
  `client_hash_id` / `content_hash_id` identifiers are used.

**Final feature table:** 92,548 content items (early-March features + March outcome label + static
content attributes).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining` -- impressions dropped from early-March to late-March within the same
content item, computed strictly from within-window observed data.

**Features:** early-March signals (`imp_early`, `clk_early`, `pos_early`, `ctr_early`) plus static
`dim_content` attributes (`content_age_days` as of March 1, `word_count`, `char_count`, `backlinks`,
`search_volume`, `competition`, `main_intent`, `content_type`, `category_count`), with missingness
flags for attributes with real gaps (~30% missing word/char_count, ~39% missing backlinks --
imputed with median, flagged rather than silently zero-filled).

**Baseline:** a single-condition, human-readable rule -- pages in the worst 20% of early-March
position are flagged for review (`weak_early_position`). Two candidate signals (volume, position)
were first audited independently with bucketed verdict tables before the

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| K | Base rate | Baseline (position rule) | Random Forest |
|---|---|---|---|
| 20 | 0.370 | 0.250 | **0.650** |
| 50 | 0.370 | 0.320 | **0.580** |
| 100 | 0.370 | 0.360 | **0.600** |

The Random Forest model reached precision@20 of 0.650 on a client-grouped test split -- roughly
2.6x the baseline's 0.250 on the same split, same metric, same seed.

**Feature importance:** the model's top predictors were `content_age_days` and `char_count`,
outranking the original early-window search signals (`ctr_early`, `pos_early`). Investigating
`content_age_days` further revealed an inverted-U relationship with decline rate (risk peaks around
82-130 days old, then falls for older content) -- explained by survivorship bias: very old pages
still present in the snapshot already proved durable, since fragile old pages were likely already
pruned before the data was captured. This was checked against a batch-import artifact explanation
and ruled out (counts were smoothly distributed across age bins, not spiked).

## 5. Limitations

*What this work cannot claim.*

- **One split, one time period:** all results come from a single grouped train/test split (8 held-out
  test clients) on March 2026 data. This is a measured, out-of-sample result -- not a guarantee that
  precision holds for every client, every month, or after the portfolio composition shifts.
- **Correlational, not causal:** this is an observational study. "Old, thin content is flagged more
  often" is an observed association, not evidence that age or length *causes* decline, and refreshing
  a flagged page is not guaranteed to reverse a decline.
- **Survivorship bias in the age signal:** the age-risk pattern likely reflects which old pages
  happened to survive into this snapshot, not a universal aging effect -- this was investigated and
  is a directional finding, not a settled one.
- **A known blind spot:** the model's most confident wrong predictions were all zero-early-click,
  moderate-impression pages it flagged as declining but that were actually stable. Any page flagged
  primarily for zero clicks deserves extra human scrutiny before action.
- **Precision degrades quickly past the top of the queue:** by rank ~100, precision is close to the
  base rate -- the ranking is only strongly trustworthy near the top.
- **Decision-support only:** every claim in this paper should be read as "these pages look worth
  reviewing first, because..." -- never as "this page will decline" or "refreshing this page will
  fix it." No causal or forward-looking guarantee is being made.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Output from the Week-7 action playbook, scored across the full 92,548-item dataset:

| Tier | Count | Basis |
|---|---|---|
| REVIEW_FIRST | 20 | Top-ranked; matches the measured precision@20 = 0.650 band |
| REVIEW_SOON | 80 | Ranks 21-100; precision degrades toward the base rate here |
| MONITOR_ONLY | 92,448 | Below any measured precision band -- score alone should not drive action |

Each item carries a reason code (`AGE_RISK_WINDOW`, `ZERO_CLICKS`, `WEAK_POSITION`, `THIN_CONTENT`,
or `LOW_SIGNAL`) so a human reviewer sees *why* a page was flagged, not just that it was.

**Human review is required before any action** -- this queue narrows attention, it does not execute
decisions. Items with a `ZERO_CLICKS` reason code need particular scrutiny given the documented blind
spot above. No automated content edits, deletions, or client-facing claims should be generated from
this queue directly.

**Monitoring:** realized precision@20 should be tracked monthly against the 0.650 baseline; a
sustained drop, a shift in the base decline rate, or a major change in portfolio composition are all
retrain triggers.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three charts, each with one clear message, already generated and committed to work/figures/ in
Week 7 / the paper-prep session: model vs baseline precision@K, the client-memorization gap
(random vs grouped split), and top feature importances. Displayed below for confirmation.

In [2]:
from IPython.display import Image, display

base_url = "https://raw.githubusercontent.com/MuhammadBilalFarooq/Assignment1/main/work/figures/"

print("Chart 1: Model vs Baseline")
display(Image(url=base_url + "model_vs_baseline.png"))

print("\nChart 2: Client Memorization Gap (Random vs Grouped Split)")
display(Image(url=base_url + "split_comparison.png"))

print("\nChart 3: Feature Importance")
display(Image(url=base_url + "feature_importance.png"))

Chart 1: Model vs Baseline



Chart 2: Client Memorization Gap (Random vs Grouped Split)



Chart 3: Feature Importance


## Self-check

Before you submit, confirm each line honestly:

Every section above is filled — ✅
Notebook runs top to bottom with no errors — do a final Runtime → Run all first, then check this
No client names, URLs, or private queries — ✅ (only hashed IDs throughout)
Claims use careful words (observed, measured, directional, decision-support) — ✅
Committed to repo — do after Run all
All 9 deployed-paper sections including Abstract and Acknowledgments — leave unchecked for now, since the actual deployed paper doesn't exist yet (this notebook only has 7 sections; the paper itself needs all 9, including Abstract and Acknowledgments, which we'll build into the HTML page)
ML-12 pieces (demo outline, social post, employer summary) — do next, in new cells below the self-check


## ML-12 — Demo, Social Post, Employer Summary

### 5-Minute Demo Outline
1. **The problem (30s):** a content team can't manually review every page every month -- which pages should they check first?
2. **The naive approach (45s):** show the baseline rule (worst 20% of position) -- precision@20 = 0.250, barely above the 0.370 base rate on this split.
3. **The model (60s):** show the Random Forest result -- precision@20 = 0.650, a 2.6x lift. Show the model-vs-baseline chart.
4. **The honesty check (90s):** show the split comparison chart -- random split scores 0.88, grouped split scores 0.58. Explain why the grouped number is the one to trust.
5. **What drives it (45s):** show the feature importance chart -- content age and length matter more than raw search signals, with the survivorship-bias caveat explained in one sentence.
6. **The output (30s):** show the ranked action queue with reason codes -- decision-support, not automation.

### Social Post
"Built a content-decline predictor on FlyRank's 79M-row search dataset. A simple rule barely beat
random guessing (25% precision). Adding a validated Random Forest model got 65% precision@20 --
but only after catching a client-memorization bug that would've overstated the result by 30 points.
Full writeup: [link]"

### Employer 3-Sentencer
I built a content-decline prioritization model on a 79-million-row real-world search performance
dataset, using a client-grouped validation design to avoid a measured 30-point overfitting trap.
The final Random Forest model beat a transparent rule-based baseline by 2.6x on precision@20 while
staying fully leakage-checked and reproducible. The output is a ranked, reason-coded action queue
designed for human review rather than automation, with documented limitations and honest,
decision-support-level claims throughout.